In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

ROOT = "/content/drive/MyDrive/ARVisionGold"
!pip -q install pyyaml

import os, yaml, json

# ensure dirs
os.makedirs(f"{ROOT}/artifacts/signals", exist_ok=True)

# load configs
with open(f"{ROOT}/configs/paths.yaml") as f:
    P = yaml.safe_load(f)
with open(f"{ROOT}/configs/signals.yaml") as f:
    SIG = yaml.safe_load(f)

CANDLE_TXT = P["artifacts"]["candle_type"]           # /artifacts/cv/candle_type.txt
LAST_SIGNAL_TXT = f"{ROOT}/artifacts/signals/last_signal.txt"
print("Paths loaded ✔")


Mounted at /content/drive
Paths loaded ✔


In [2]:
def load_candle_type(path):
    try:
        with open(path) as f:
            return f.read().strip().upper()
    except FileNotFoundError:
        return None

def fuse_signal(model_label:str, model_prob:float, candle_type:str, cfg:dict):
    # normalize
    mlabel = (model_label or "").upper()
    ctype  = (candle_type or "").upper()

    bull_words = set(map(str.upper, cfg["fusion"]["bullish_words"]))   # ["UP","BULLISH"]
    bear_words = set(map(str.upper, cfg["fusion"]["bearish_words"]))   # ["DOWN","BEARISH"]
    th = cfg["thresholds"]

    up = mlabel in bull_words
    dn = mlabel in bear_words
    ct_bull = (ctype == "BULLISH")
    ct_bear = (ctype == "BEARISH")

    # hard signals
    if up and model_prob >= th["rf_prob_buy"] and ct_bull:
        return "STRONG BUY"
    if dn and model_prob >= th["rf_prob_sell"] and ct_bear:
        return "STRONG SELL"

    # neutral zone
    if model_prob < th["no_trade_band"]:
        return "WAIT"

    # soft signals if outside neutral band
    if up: return "BUY"
    if dn: return "SELL"
    return "WAIT"


In [3]:
candle_type = load_candle_type(CANDLE_TXT)  # requires Module-2 output; else None
tests = [
    ("UP",   0.82, candle_type),
    ("DOWN", 0.75, candle_type),
    ("UP",   0.45, candle_type),
    ("DOWN", 0.52, candle_type),
]
for lab, p, ct in tests:
    print(lab, p, ct, "=>", fuse_signal(lab, p, ct, SIG))


UP 0.82 BULLISH => STRONG BUY
DOWN 0.75 BULLISH => SELL
UP 0.45 BULLISH => WAIT
DOWN 0.52 BULLISH => SELL


In [4]:
sample_label, sample_prob = "UP", 0.72
ctype = load_candle_type(CANDLE_TXT)
final_signal = fuse_signal(sample_label, sample_prob, ctype, SIG)

with open(LAST_SIGNAL_TXT, "w") as f:
    f.write(final_signal)

print("Final signal (sample):", final_signal)
print("Written:", LAST_SIGNAL_TXT)


Final signal (sample): STRONG BUY
Written: /content/drive/MyDrive/ARVisionGold/artifacts/signals/last_signal.txt
